# 卷积神经网络 CNN

按代码单元格的原编号查阅课程手册。

In [ ]:
import os
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

tf.keras.utils.set_random_seed(42)
device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
output_dir = Path(os.environ.get('COURSE_OUTPUT_DIR', 'outputs'))
output_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv('15min_in.csv', usecols=['西直门'], encoding='gbk')
y = df['西直门'].to_numpy(dtype=float)
print(f'Device: {device}, series length: {len(y)}')
plt.figure(figsize=(12, 4))
plt.grid(True)
plt.plot(y)
plt.show()

In [ ]:
# 最后 300 点作为测试集，只用训练段拟合缩放参数。
test_size = 300
train_iter = y[:-test_size]
test_iter = y[-test_size:]
scaler = MinMaxScaler(feature_range=(-1, 1))
train_norm = scaler.fit_transform(train_iter.reshape(-1, 1))
train_set = tf.convert_to_tensor(train_norm.ravel(), dtype=tf.float32)

In [ ]:
# 定义时间窗口
Time_window_size = 72

# 从原时间序列中抽取出训练样本，用第1个值到第72个值作为X输入，预测第73个值作为y输出，这是一个用于训练的数据点，时间窗口向后滑动以此类推
def input_data(seq, ws):
    out = []
    L = len(seq)
    for i in range(L-ws):
        window = seq[i:i + ws]
        label = seq[i + ws:i + ws + 1]
        out.append((window, label))
    return out


train_data = input_data(train_set, Time_window_size)
len(train_data)  # 等于1800（原始数据集长度）-300（测试集长度）-72（时间窗口）

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
output = sliding_window_view(train_set.numpy(), Time_window_size + 1)
print(f'Train windows: {len(output)}')

In [ ]:
class CNNnetwork(tf.keras.Sequential):
    def __init__(self):
        super().__init__([
            layers.Conv1D(64, kernel_size=2, activation='relu'),
            layers.Conv1D(32, kernel_size=2, activation='relu'),
            layers.MaxPooling1D(pool_size=2, strides=2),
            layers.Dropout(0.5),
            layers.Flatten(),
            layers.Dense(640, activation='relu'),
            layers.Dense(1),
        ])


net = CNNnetwork()

In [ ]:
criterion = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005, epsilon=1e-8)

In [ ]:
@tf.function
def train_1d_step(seq, target):
    with tf.device(device), tf.GradientTape() as tape:
        prediction = net(seq[None, :, None], training=True)
        loss = criterion(target[None, :], prediction)
    gradients = tape.gradient(loss, net.trainable_variables)
    optimizer.apply_gradients(zip(gradients, net.trainable_variables))
    return loss


epochs = 20
for epoch in range(epochs):
    total_loss = sum(float(train_1d_step(seq, target)) for seq, target in train_data)
    print(f'Epoch: {epoch + 1:2} Mean loss: {total_loss / len(train_data):.8f}')

In [ ]:
future = len(test_iter)
preds = train_set[-Time_window_size:].numpy().tolist()
with tf.device(device):
    for _ in range(future):
        seq = tf.constant(preds[-Time_window_size:], dtype=tf.float32)[None, :, None]
        preds.append(float(net(seq, training=False)[0, 0]))
true_predictions = scaler.inverse_transform(np.array(preds[Time_window_size:]).reshape(-1, 1)).ravel()
errors = true_predictions - test_iter
last_value = train_iter[-1]
print(f'Recursive test steps: {future}')
print(f'Recursive test MAE: {np.abs(errors).mean():.6f}')
print(f'Recursive test RMSE: {np.sqrt(np.mean(errors ** 2)):.6f}')
print(f'Fixed origin last observation RMSE: {np.sqrt(np.mean((last_value - test_iter) ** 2)):.6f}')

In [ ]:
plt.figure(figsize=(12, 4))
plt.grid(True)
plt.plot(y, label='Observed')
plt.plot(np.arange(len(train_iter), len(y)), true_predictions, label='Recursive forecast')
plt.legend()
plt.show()

In [ ]:
from pathlib import Path
import struct

import numpy as np
import tensorflow as tf


def load_mnist(prefix):
    root = Path('Datasets/MNIST/MNIST/raw')
    with (root / f'{prefix}-images-idx3-ubyte').open('rb') as images:
        magic, count, height, width = struct.unpack('>IIII', images.read(16))
        if (magic, height, width) != (2051, 28, 28):
            raise ValueError('Invalid MNIST image header')
        pixels = np.frombuffer(images.read(), dtype=np.uint8).reshape(count, height, width, 1)
    with (root / f'{prefix}-labels-idx1-ubyte').open('rb') as labels:
        if struct.unpack('>II', labels.read(8)) != (2049, count):
            raise ValueError('Invalid MNIST label header')
        classes = np.frombuffer(labels.read(), dtype=np.uint8)
    if len(classes) != count:
        raise ValueError('Truncated MNIST labels')
    return tf.data.Dataset.from_tensor_slices((pixels, classes))


def normalize_images(images, labels):
    return tf.cast(images, tf.float32) / 127.5 - 1.0, tf.cast(labels, tf.int64)


batch_size = 64
train_dataset = load_mnist('train')
test_dataset = load_mnist('t10k')
train_loader = train_dataset.shuffle(len(train_dataset), seed=42).batch(batch_size).map(normalize_images)
test_loader = test_dataset.batch(batch_size).map(normalize_images)

In [ ]:
from tensorflow.keras import layers

num_classes = 10
lr = 0.001
epochs = 20
device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'


class ConvModule(tf.keras.Sequential):
    def __init__(self):
        super().__init__([
            layers.Input((28, 28, 1)),
            layers.Conv2D(32, kernel_size=3, padding='same'),
            layers.MaxPooling2D(2, 2),
            layers.ReLU(),
            layers.Conv2D(64, kernel_size=3, padding='same'),
            layers.MaxPooling2D(2, 2),
            layers.ReLU(),
            layers.Flatten(),
            layers.Dense(1024, activation='relu'),
            layers.Dense(512, activation='relu'),
            layers.Dense(num_classes),
        ])


with tf.device(device):
    net = ConvModule()
    criterion = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, epsilon=1e-8)

In [ ]:
@tf.function(reduce_retracing=True)
def train_mnist_batch(model, inputs, labels):
    with tf.GradientTape() as tape:
        output = model(inputs, training=True)
        loss = criterion(labels, output)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss, output


def _mnist_epoch(model, data_loader, device, *, training):
    total_loss, correct, count = 0., 0, 0
    with tf.device(device):
        for inputs, labels in data_loader:
            if training:
                loss, output = train_mnist_batch(model, inputs, labels)
            else:
                output = model(inputs, training=False)
                loss = criterion(labels, output)
            size = int(tf.size(labels))
            count += size
            total_loss += float(loss) * size
            matches = tf.argmax(output, axis=1, output_type=labels.dtype) == labels
            correct += int(tf.math.count_nonzero(matches))
    return total_loss / count, correct / count


def train_epoch(net, data_loader, device):
    return _mnist_epoch(net, data_loader, device, training=True)


def test_epoch(net, data_loader, device):
    return _mnist_epoch(net, data_loader, device, training=False)

In [ ]:
# 存储每一个epoch的loss与acc的变化，便于后面的可视化
train_loss_list = []
train_acc_list = []
test_loss_list = []
test_acc_list = []

# 进行训练
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(net, data_loader=train_loader, device=device)
    test_loss, test_acc = test_epoch(net, data_loader=test_loader, device=device)

    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)
    test_loss_list.append(test_loss)
    test_acc_list.append(test_acc)
    print('epoch %d, train_loss %.6f, train_acc %.6f' % (epoch + 1, train_loss, train_acc))
    print('test_loss %.6f , test_acc %.6f' % (test_loss, test_acc)) 


In [ ]:
for name, train_values, test_values in [
    ('Loss', train_loss_list, test_loss_list),
    ('Acc', train_acc_list, test_acc_list),
]:
    fig, ax = plt.subplots()
    epochs_run = np.arange(1, len(train_values) + 1)
    ax.plot(epochs_run, train_values, label='train')
    ax.plot(epochs_run, test_values, label='test')
    ax.set(xlabel='Epoch', ylabel=name)
    ax.legend()
    fig.savefig(output_dir / f'{name}.jpg', dpi=150)
    plt.show()